# Feito para carregar os dados no banco:

In [1]:
# --- Preencher SQLite em lotes de ~100 dias úteis por vez (retomável) ---
# Requisitos (uma vez):  pip install yfinance pandas pandas-datareader

import os, sqlite3, time, math, datetime as dt
from contextlib import closing
import pandas as pd

def carregar_dados(DB_PATH, TICKERS, START, BATCH_N, SLEEP_BETWEEN, MAX_RETRIES):
    # -- criação de tabelas (mesmo esquema da app) --
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    DDL_PRECO = """
    CREATE TABLE IF NOT EXISTS preco_diario (
        id INTEGER PRIMARY KEY,
        ticker TEXT NOT NULL,
        date   DATE NOT NULL,
        open REAL, high REAL, low REAL, close REAL, adj_close REAL, volume INTEGER,
        created_at DATETIME DEFAULT (datetime('now'))
    );
    """
    DDL_UQ = "CREATE UNIQUE INDEX IF NOT EXISTS uq_ticker_date ON preco_diario(ticker, date);"
    with closing(conn.cursor()) as cur:
        cur.execute(DDL_PRECO); cur.execute(DDL_UQ); conn.commit()

    def get_db_span(ticker):
        with closing(conn.cursor()) as cur:
            cur.execute("SELECT COUNT(*), MIN(date), MAX(date) FROM preco_diario WHERE ticker=?;", (ticker,))
            total, mind, maxd = cur.fetchone()
        return total, mind, maxd

    import pandas as pd

    def _normalize_ohlcv(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
        """
        Converte qualquer retorno (Yahoo multi-index, Stooq, etc.)
        para colunas simples: Open, High, Low, Close, Adj_Close, Volume.
        """
        if df is None or df.empty:
            return pd.DataFrame()

        # 1) Se vier MultiIndex (ex.: ('Open','NVDA')), reduz para colunas simples
        if isinstance(df.columns, pd.MultiIndex):
            # tenta selecionar o nível que tem o ticker
            upper = ticker.upper()
            selected = None
            for level in range(df.columns.nlevels - 1, -1, -1):
                vals = [str(v).upper() for v in df.columns.get_level_values(level)]
                if upper in vals:
                    selected = level
                    break
            if selected is not None:
                # pega apenas o subdataframe daquele ticker
                try:
                    df = df.xs(upper, axis=1, level=selected, drop_level=True)
                except Exception:
                    # se não deu para dropar o nível, tenta reordenar
                    df = df.swaplevel(selected, 0, axis=1)
                    df = df[upper]
            else:
                # flattener genérico: junta níveis
                df.columns = ['_'.join([str(x) for x in c if x is not None]) for c in df.columns]

        # 2) Padroniza nomes
        rename = {}
        for c in df.columns:
            lc = str(c).lower().strip()
            if lc in ("open", "high", "low", "close", "volume", "adj close", "adj_close", "adjclose"):
                if "adj" in lc:
                    rename[c] = "Adj_Close"
                elif lc == "volume":
                    rename[c] = "Volume"
                else:
                    rename[c] = lc.capitalize()
        df = df.rename(columns=rename)

        # 3) Garante colunas essenciais e ordem
        if "Adj_Close" not in df.columns and "Close" in df.columns:
            df["Adj_Close"] = df["Close"]

        keep = [c for c in ["Open", "High", "Low", "Close", "Adj_Close", "Volume"] if c in df.columns]
        df = df[keep].copy()

        # 4) Limpa NaNs de Close
        if "Close" in df.columns:
            df = df.dropna(subset=["Close"])

        # index como datetime
        df.index = pd.to_datetime(df.index)

        return df

    def upsert_df(df, ticker):
        df = _normalize_ohlcv(df, ticker)
        if df.empty:
            return 0

        rows = []
        for idx, r in df.iterrows():
            rows.append({
                "ticker": ticker,
                "date": pd.Timestamp(idx).date().isoformat(),
                "open":  float(r["Open"])  if "Open"  in df.columns and pd.notna(r["Open"])  else None,
                "high":  float(r["High"])  if "High"  in df.columns and pd.notna(r["High"])  else None,
                "low":   float(r["Low"])   if "Low"   in df.columns and pd.notna(r["Low"])   else None,
                "close": float(r["Close"]) if "Close" in df.columns and pd.notna(r["Close"]) else None,
                "adj_close": float(r["Adj_Close"]) if "Adj_Close" in df.columns and pd.notna(r["Adj_Close"])
                            else (float(r["Close"]) if "Close" in df.columns and pd.notna(r["Close"]) else None),
                "volume": int(r["Volume"]) if "Volume" in df.columns and pd.notna(r["Volume"]) else 0,
            })

        sql = """
        INSERT INTO preco_diario (ticker, date, open, high, low, close, adj_close, volume)
        VALUES (:ticker, :date, :open, :high, :low, :close, :adj_close, :volume)
        ON CONFLICT(ticker, date) DO UPDATE SET
        open=excluded.open, high=excluded.high, low=excluded.low, close=excluded.close,
        adj_close=excluded.adj_close, volume=excluded.volume;
        """
        with closing(conn.cursor()) as cur:
            cur.executemany(sql, rows)
        conn.commit()
        return len(rows)

    def next_window(start_date_str, batch_n):
        """Retorna (d0, d1) cobrindo ~batch_n dias úteis a partir de d0."""
        d0 = pd.to_datetime(start_date_str).date()
        bdays = pd.bdate_range(d0, periods=batch_n)
        d1 = bdays[-1].date() if len(bdays) else d0
        # garante que não passe de hoje
        today = pd.Timestamp.today().date()
        if d1 > today: d1 = today
        return d0, d1

    def fetch_yahoo(ticker, d0, d1):
        import yfinance as yf
        # Yahoo usa end exclusivo → somar 1 dia
        end_exc = pd.Timestamp(d1) + pd.Timedelta(days=1)
        df = yf.download(ticker, start=str(d0), end=str(end_exc.date()),
                        auto_adjust=False, progress=False, threads=False)
        if df is not None and not df.empty:
            return df
        # fallback 2
        return yf.Ticker(ticker).history(start=str(d0), end=str(end_exc.date()),
                                        interval="1d", auto_adjust=False)

    def fetch_stooq(ticker, d0, d1):
        try:
            from pandas_datareader import data as pdr
        except Exception:
            return pd.DataFrame()
        df = pdr.DataReader(ticker, "stooq", start=d0, end=d1)
        if df is not None and not df.empty:
            df = df.sort_index()
        return df

    def fetch_block_resilient(ticker, d0, d1, max_retries=MAX_RETRIES):
        last_exc = None
        for i in range(max_retries):
            try:
                df = fetch_yahoo(ticker, d0, d1)
                if df is not None and not df.empty:
                    return df
                last_exc = RuntimeError("Yahoo vazio")
            except Exception as e:
                last_exc = e
            # backoff (mais longo para 429/Rate)
            s = 1.5 * (2 ** i)
            print(f"[INFO] retry {i+1}/{max_retries} para {ticker} {d0}->{d1} (aguardando {s:.1f}s) | {last_exc}")
            time.sleep(s)
        # Yahoo falhou → Stooq
        df = fetch_stooq(ticker, d0, d1)
        if df is not None and not df.empty:
            print("[INFO] usando Stooq como fallback")
            return df
        raise last_exc if last_exc else RuntimeError("Falha desconhecida")

    def fill_ticker(ticker, start_str, batch_n=BATCH_N):
        # define ponto de partida: MAX(date)+1 ou START
        total, mind, maxd = get_db_span(ticker)
        if maxd:
            start = (pd.to_datetime(maxd).date() + dt.timedelta(days=1)).isoformat()
            if pd.to_datetime(start) < pd.to_datetime(start_str):
                start = start_str
        else:
            start = start_str

        today = pd.Timestamp.today().date()
        if pd.to_datetime(start).date() > today:
            print(f"[{ticker}] nada a fazer (start {start} > hoje)")
            return 0

        print(f"[{ticker}] iniciando em {start} até {today} (lotes de {batch_n} pregões)")
        inserted_total = 0
        cur = start
        while True:
            d0, d1 = next_window(cur, batch_n)
            if d0 > today: break
            try:
                df = fetch_block_resilient(ticker, d0, d1)
            except Exception as e:
                print(f"[{ticker}] ERRO ao baixar {d0}->{d1}: {e}")
                break
            n = upsert_df(df, ticker)
            inserted_total += n
            print(f"[{ticker}] +{n:4d} linhas | {d0} -> {d1}")
            # próximo lote: próximo dia útil após d1
            cur = (pd.bdate_range(d1, periods=2)[-1].date()).isoformat()
            if pd.to_datetime(cur).date() > today:
                break
            time.sleep(SLEEP_BETWEEN)
        # resumo
        tot, mi, mx = get_db_span(ticker)
        print(f"[{ticker}] DONE | inseridas agora: {inserted_total} | total na base: {tot} (de {mi} até {mx})")
        return inserted_total
    # ===================== EXECUTAR =====================
    for t in TICKERS:
        fill_ticker(t, START, BATCH_N)
    conn.close()

import os
from pathlib import Path

def resolve_db_path():
    uri = os.getenv("DATABASE_URL", "sqlite:///instance/app.db")
    if not uri.startswith("sqlite:///"):
        raise ValueError("Este script suporta apenas SQLite (sqlite:///...)")
    rel = uri.replace("sqlite:///", "")          # ex.: instance/app.db
    p = Path(rel)
    if p.is_absolute():                          # se já for absoluto, ok
        return p
    # torna absoluto a partir do diretório do projeto (cwd)
    return (Path.cwd() / p).resolve()

# ===================== CONFIG =====================
DB_PATH = resolve_db_path()
print(DB_PATH)
TICKERS  = ["TSLA"]         # coloque vários se quiser: ["AAPL","NVDA","MSFT"]
START    = "2015-01-01"     # data inicial (passado)
BATCH_N  = 100              # <- tamanho do lote em pregões (100 a 100)
SLEEP_BETWEEN = 1.0         # pausa entre lotes (segundos)
MAX_RETRIES   = 6           # tentativas por lote (com backoff)
carregar_dados(DB_PATH, TICKERS, START, BATCH_N, SLEEP_BETWEEN, MAX_RETRIES)
# ===================================================

C:\Users\CLIENTE\OneDrive\POS\FIAP\TECH CHALLENGE\tech_challenge_4\stock-lstm-flask\instance\app.db
[TSLA] iniciando em 2015-01-01 até 2025-10-08 (lotes de 100 pregões)
[TSLA] +  96 linhas | 2015-01-01 -> 2015-05-20
[TSLA] +  97 linhas | 2015-05-21 -> 2015-10-07
[TSLA] +  95 linhas | 2015-10-08 -> 2016-02-24
[TSLA] +  97 linhas | 2016-02-25 -> 2016-07-13
[TSLA] +  98 linhas | 2016-07-14 -> 2016-11-30
[TSLA] +  95 linhas | 2016-12-01 -> 2017-04-19
[TSLA] +  97 linhas | 2017-04-20 -> 2017-09-06
[TSLA] +  96 linhas | 2017-09-07 -> 2018-01-24
[TSLA] +  97 linhas | 2018-01-25 -> 2018-06-13
[TSLA] +  98 linhas | 2018-06-14 -> 2018-10-31
[TSLA] +  94 linhas | 2018-11-01 -> 2019-03-20
[TSLA] +  97 linhas | 2019-03-21 -> 2019-08-07
[TSLA] +  97 linhas | 2019-08-08 -> 2019-12-25
[TSLA] +  96 linhas | 2019-12-26 -> 2020-05-13
[TSLA] +  97 linhas | 2020-05-14 -> 2020-09-30
[TSLA] +  95 linhas | 2020-10-01 -> 2021-02-17
[TSLA] +  97 linhas | 2021-02-18 -> 2021-07-07
[TSLA] +  99 linhas | 2021-07-08